# 分层交通流图构建 V2

## 主要改进
1. **同位置组定义**: abs_pm 完全一致 + 经纬度距离为0 + OSRM距离为0
2. **连边验证**: 除了 PM 差值过滤，还使用 OSRM 路径距离验证（≤4 miles）
3. **可视化**: 边使用 OSRM 实际路径而非直线
4. **颜色区分**: 不同类型站点和边使用不同颜色

## 构图策略

### 第一层：ML + HV 骨干网络
- 只处理 ML 和 HV 类型
- 同位置组内不连边
- 每个 ML/HV 连接上下游最近位置组的所有 ML/HV

### 第二层：OR/FR 连接
- OR (入口): 向下游搜索，连接路径上的可连站点，直到找到 ML
- FR (出口): 向上游搜索，接受路径上的可连站点，直到找到 ML

### 第三层：FF 连接（可选）
- 使用 OSRM 查找同高速和跨高速连接

In [21]:
import pandas as pd
import numpy as np
import requests
import time
import os
import glob
import json
from datetime import datetime
from math import radians, sin, cos, sqrt, atan2
from tqdm import tqdm
import matplotlib.pyplot as plt
import folium
from folium import plugins
import warnings
warnings.filterwarnings('ignore')

# ============== 配置 ==============
META_DIR = "../d03_meta"
OUTPUT_DIR = "./output/layered_graph_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# OSRM 服务器
OSRM_SERVER = "http://localhost:5000"
RATE_LIMIT_DELAY = 0.01

# 构图参数
MAX_PM_DISTANCE = 4.0       # 最大 PM 差值（英里）- 初筛
MAX_OSRM_DISTANCE = 4.0     # 最大 OSRM 路径距离（英里）- 最终验证

# 高速公路筛选
TARGET_FWY = '99'
TARGET_DIR = 'N'

print("配置完成！")
print(f"最大PM距离 (初筛): {MAX_PM_DISTANCE} mi")
print(f"最大OSRM距离 (验证): {MAX_OSRM_DISTANCE} mi")
print(f"目标高速: {TARGET_FWY if TARGET_FWY else '全部'}")

配置完成！
最大PM距离 (初筛): 4.0 mi
最大OSRM距离 (验证): 4.0 mi
目标高速: 99


## 1. 加载元数据

In [23]:
# 查找最新的元数据文件
meta_files = glob.glob(os.path.join(META_DIR, "d03_text_meta_*.txt"))
if meta_files:
    meta_file = sorted(meta_files)[-1]
else:
    raise FileNotFoundError("未找到 D3 元数据文件")

print(f"使用元数据: {meta_file}")

META_COLUMNS = [
    'ID', 'Fwy', 'Dir', 'District', 'County', 'City',
    'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
    'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2',
    'User_ID_3', 'User_ID_4'
]

meta_df = pd.read_csv(
    meta_file, sep='\t', names=META_COLUMNS, header=0,
    dtype={'ID': str, 'Fwy': str}
)

# 坐标有效性检查
def is_valid_coord(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return False
    return (32 <= lat <= 42) and (-124 <= lon <= -114)

meta_df['Valid_Coord'] = meta_df.apply(
    lambda r: is_valid_coord(r['Latitude'], r['Longitude']), axis=1
)

meta_valid = meta_df[meta_df['Valid_Coord']].copy()

# 应用高速公路筛选
if TARGET_FWY:
    meta_valid = meta_valid[meta_valid['Fwy'] == TARGET_FWY]
if TARGET_DIR:
    meta_valid = meta_valid[meta_valid['Dir'] == TARGET_DIR]

print(f"\n筛选后站点数: {len(meta_valid)}")
print(f"\n各类型站点数:")
print(meta_valid['Type'].value_counts())

使用元数据: ../d03_meta/d03_text_meta_2025_12_30.txt

筛选后站点数: 159

各类型站点数:
Type
ML    75
OR    36
HV    28
FR    18
FF     2
Name: count, dtype: int64


## 2. OSRM 和距离计算函数

In [24]:
def calc_distance_geo(lat1, lon1, lat2, lon2):
    """Haversine 直线距离（英里）"""
    if lat1 == lat2 and lon1 == lon2:
        return 0.0
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c


def calc_osrm_route(lat1, lon1, lat2, lon2, server=OSRM_SERVER, timeout=10):
    """
    OSRM 路径查询，返回距离和路径几何
    
    返回: {
        'distance_mi': 距离（英里），
        'duration_min': 时间（分钟），
        'geometry': GeoJSON 几何对象,
        'status': 'ok' / 'no_route' / 'error',
        'error_msg': 错误信息
    }
    """
    # 如果坐标相同，直接返回0
    if lat1 == lat2 and lon1 == lon2:
        return {
            'distance_mi': 0.0,
            'duration_min': 0.0,
            'geometry': None,
            'status': 'ok',
            'error_msg': None
        }
    
    url = f"{server}/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
    params = {
        "overview": "full",
        "geometries": "geojson",
        "steps": "false"
    }
    
    try:
        response = requests.get(url, params=params, timeout=timeout)
        data = response.json()
        
        if data.get("code") != "Ok":
            return {
                'distance_mi': None,
                'duration_min': None,
                'geometry': None,
                'status': 'no_route',
                'error_msg': data.get("code", "Unknown")
            }
        
        route = data["routes"][0]
        return {
            'distance_mi': route["distance"] / 1609.34,
            'duration_min': route["duration"] / 60,
            'geometry': route["geometry"],
            'status': 'ok',
            'error_msg': None
        }
        
    except Exception as e:
        return {
            'distance_mi': None,
            'duration_min': None,
            'geometry': None,
            'status': 'error',
            'error_msg': str(e)
        }


# 测试 OSRM 连接
print("测试 OSRM 服务器连接...")
test_result = calc_osrm_route(38.5, -121.5, 38.6, -121.4, OSRM_SERVER)
if test_result['status'] == 'ok':
    print(f"✓ OSRM 服务器正常，测试距离: {test_result['distance_mi']:.2f} mi")
    OSRM_AVAILABLE = True
else:
    print(f"✗ OSRM 连接失败: {test_result['error_msg']}")
    print("将跳过 OSRM 验证，仅使用 PM 距离过滤")
    OSRM_AVAILABLE = False

测试 OSRM 服务器连接...
✓ OSRM 服务器正常，测试距离: 12.80 mi


## 3. 方向与下游定义

高速公路方向与 abs_pm 关系：
- **N (北向) / E (东向)**: 交通流方向与 abs_pm 递增方向一致，下游 = PM 更大
- **S (南向) / W (西向)**: 交通流方向与 abs_pm 递减方向一致，下游 = PM 更小

## 4. 同位置组构建（严格定义）

In [25]:
def build_location_groups_strict(sorted_group, check_osrm=True, osrm_server=OSRM_SERVER):
    """
    将站点按严格条件划分为同位置组
    
    同位置条件（必须全部满足）:
    1. Abs_PM 完全一致
    2. 经纬度距离为 0（或非常接近）
    3. OSRM 距离为 0（可选）
    
    返回: [(group_pm, [station_indices]), ...]
    """
    if len(sorted_group) == 0:
        return []
    
    groups = []
    current_group = [0]
    group_pm = sorted_group.iloc[0]['Abs_PM']
    group_lat = sorted_group.iloc[0]['Latitude']
    group_lon = sorted_group.iloc[0]['Longitude']
    
    for i in range(1, len(sorted_group)):
        row = sorted_group.iloc[i]
        pm = row['Abs_PM']
        lat = row['Latitude']
        lon = row['Longitude']
        
        # 条件1: Abs_PM 完全一致
        if pm != group_pm:
            groups.append((group_pm, current_group))
            current_group = [i]
            group_pm = pm
            group_lat = lat
            group_lon = lon
            continue
        
        # 条件2: 经纬度距离为 0 或非常接近（< 0.01 mi ≈ 16m）
        geo_dist = calc_distance_geo(group_lat, group_lon, lat, lon)
        if geo_dist > 0.01:
            groups.append((group_pm, current_group))
            current_group = [i]
            group_pm = pm
            group_lat = lat
            group_lon = lon
            continue
        
        # 条件3: OSRM 距离为 0（可选）
        if check_osrm and OSRM_AVAILABLE:
            osrm_result = calc_osrm_route(group_lat, group_lon, lat, lon, osrm_server)
            if osrm_result['status'] == 'ok' and osrm_result['distance_mi'] > 0.01:
                groups.append((group_pm, current_group))
                current_group = [i]
                group_pm = pm
                group_lat = lat
                group_lon = lon
                time.sleep(RATE_LIMIT_DELAY)
                continue
            time.sleep(RATE_LIMIT_DELAY)
        
        # 满足所有条件，加入当前组
        current_group.append(i)
    
    groups.append((group_pm, current_group))
    return groups


def get_types_in_group(sorted_group, station_indices):
    """获取位置组中的站点类型集合"""
    return set(sorted_group.iloc[i]['Type'] for i in station_indices)


def has_ml_in_group(sorted_group, station_indices):
    """检查位置组是否包含 ML"""
    return 'ML' in get_types_in_group(sorted_group, station_indices)


print("同位置组构建函数定义完成")

同位置组构建函数定义完成


In [26]:
# 测试同位置组划分
print("测试同位置组划分...")
test_fwy_dir = meta_valid.groupby(['Fwy', 'Dir']).first().index[0]
test_group = meta_valid[(meta_valid['Fwy'] == test_fwy_dir[0]) & 
                        (meta_valid['Dir'] == test_fwy_dir[1])].sort_values('Abs_PM').reset_index(drop=True)

# 先不检查 OSRM，快速测试
test_loc_groups = build_location_groups_strict(test_group, check_osrm=False)
print(f"测试 Fwy {test_fwy_dir[0]}-{test_fwy_dir[1]}:")
print(f"  站点数: {len(test_group)}")
print(f"  位置组数: {len(test_loc_groups)}")
print(f"  多站点组数: {sum(1 for _, stations in test_loc_groups if len(stations) > 1)}")

# 显示多站点组
print(f"\n多站点位置组示例:")
multi_count = 0
for pm, stations in test_loc_groups:
    if len(stations) > 1:
        print(f"  PM={pm:.3f}: {len(stations)} 站点")
        for idx in stations:
            s = test_group.iloc[idx]
            print(f"    - {s['ID']} ({s['Type']}) lat={s['Latitude']:.6f} lon={s['Longitude']:.6f}")
        multi_count += 1
        if multi_count >= 3:
            break

测试同位置组划分...
测试 Fwy 99-N:
  站点数: 159
  位置组数: 117
  多站点组数: 24

多站点位置组示例:
  PM=275.400: 2 站点
    - 319091 (ML) lat=38.256213 lon=-121.294616
    - 319092 (OR) lat=38.256213 lon=-121.294616
  PM=281.669: 2 站点
    - 3027042 (FR) lat=38.341309 lon=-121.334507
    - 3027041 (ML) lat=38.341309 lon=-121.334507
  PM=284.660: 2 站点
    - 317143 (ML) lat=38.377839 lon=-121.363259
    - 317142 (OR) lat=38.377839 lon=-121.363259


## 4. 边记录创建（包含 OSRM 路径）

In [27]:
def create_edge_with_osrm(source, target, fwy, direction, edge_type, 
                          source_g_idx, target_g_idx, osrm_server=OSRM_SERVER):
    """
    创建边记录，包含 OSRM 路径信息
    
    返回: (edge_record, is_valid)
    - edge_record: 边信息字典
    - is_valid: 是否通过 OSRM 距离验证
    """
    # 计算直线距离
    geo_dist = calc_distance_geo(
        source['Latitude'], source['Longitude'],
        target['Latitude'], target['Longitude']
    )
    
    # 获取 OSRM 路径
    osrm_result = None
    osrm_dist = None
    osrm_geometry = None
    
    if OSRM_AVAILABLE:
        osrm_result = calc_osrm_route(
            source['Latitude'], source['Longitude'],
            target['Latitude'], target['Longitude'],
            osrm_server
        )
        if osrm_result['status'] == 'ok':
            osrm_dist = osrm_result['distance_mi']
            osrm_geometry = json.dumps(osrm_result['geometry']) if osrm_result['geometry'] else None
        time.sleep(RATE_LIMIT_DELAY)
    
    # 验证距离是否在阈值内
    is_valid = True
    if osrm_dist is not None and osrm_dist > MAX_OSRM_DISTANCE:
        is_valid = False
    
    edge_record = {
        'Source_ID': source['ID'],
        'Target_ID': target['ID'],
        'Fwy': fwy,
        'Dir': direction,
        'Source_Type': source['Type'],
        'Target_Type': target['Type'],
        'Source_PM': source['Abs_PM'],
        'Target_PM': target['Abs_PM'],
        'PM_Diff': abs(target['Abs_PM'] - source['Abs_PM']),
        'Source_Lat': source['Latitude'],
        'Source_Lon': source['Longitude'],
        'Target_Lat': target['Latitude'],
        'Target_Lon': target['Longitude'],
        'Source_Name': source['Name'],
        'Target_Name': target['Name'],
        'Edge_Type': edge_type,
        'Source_Group_Idx': source_g_idx,
        'Target_Group_Idx': target_g_idx,
        'Geo_Distance': geo_dist,
        'OSRM_Distance': osrm_dist,
        'OSRM_Geometry': osrm_geometry,
    }
    
    return edge_record, is_valid


print("边创建函数定义完成")

边创建函数定义完成


## 5. 第一层：ML + HV 骨干网络

In [28]:
def is_downstream_direction(direction):
    """
    判断交通流下游方向与 abs_pm 的关系
    
    - N (北向) / E (东向): 交通流向北/向东，abs_pm 递增方向是下游
    - S (南向) / W (西向): 交通流向南/向西，abs_pm 递减方向是下游
    
    返回: True 表示 abs_pm 递增是下游，False 表示 abs_pm 递减是下游
    """
    return direction in ['N', 'E']


def build_layer1_backbone(meta_df, max_pm_distance=4.0, check_osrm_colocation=False):
    """
    第一层：构建 ML + HV 骨干网络
    
    规则:
    1. 只处理 ML 和 HV 类型
    2. 同位置组内不连边（严格定义：PM相同 + 坐标相同）
    3. 每个 ML/HV 连接下游最近位置组的所有 ML/HV
    4. 跳过只包含 OR/FR/FF 的位置组
    5. 使用 OSRM 验证距离
    
    方向处理:
    - N/E 方向: abs_pm 递增是下游
    - S/W 方向: abs_pm 递减是下游
    """
    all_edges = []
    all_groups = []
    rejected_edges = []  # 记录被 OSRM 距离拒绝的边
    
    for (fwy, direction), fwy_group in tqdm(meta_df.groupby(['Fwy', 'Dir']), desc="Layer1: ML+HV骨干"):
        sorted_group = fwy_group.sort_values('Abs_PM').reset_index(drop=True)
        n = len(sorted_group)
        
        if n < 2:
            continue
        
        # 判断下游方向
        pm_increasing_is_downstream = is_downstream_direction(direction)
        
        # 划分同位置组（严格定义）
        location_groups = build_location_groups_strict(sorted_group, check_osrm=check_osrm_colocation)
        n_groups = len(location_groups)
        
        # 记录位置组信息
        for g_idx, (g_pm, g_stations) in enumerate(location_groups):
            types_in_group = get_types_in_group(sorted_group, g_stations)
            all_groups.append({
                'Fwy': fwy,
                'Dir': direction,
                'Group_Idx': g_idx,
                'Group_PM': g_pm,
                'N_Stations': len(g_stations),
                'Types': '+'.join(sorted(types_in_group)),
                'Has_ML': 'ML' in types_in_group,
                'Has_HV': 'HV' in types_in_group,
                'Station_IDs': ','.join(sorted_group.iloc[i]['ID'] for i in g_stations)
            })
        
        # 找出包含 ML 或 HV 的位置组索引
        ml_hv_group_indices = []
        for g_idx, (g_pm, g_stations) in enumerate(location_groups):
            types = get_types_in_group(sorted_group, g_stations)
            if 'ML' in types or 'HV' in types:
                ml_hv_group_indices.append(g_idx)
        
        # 为每个 ML/HV 位置组构建连边
        for i, g_idx in enumerate(ml_hv_group_indices):
            g_pm, g_stations = location_groups[g_idx]
            
            # 根据方向确定上下游
            if pm_increasing_is_downstream:
                # N/E 方向: PM递增是下游
                # 下游 = 索引更大 (PM更大)
                # 上游 = 索引更小 (PM更小)
                downstream_g_idx = ml_hv_group_indices[i + 1] if i < len(ml_hv_group_indices) - 1 else None
                upstream_g_idx = ml_hv_group_indices[i - 1] if i > 0 else None
            else:
                # S/W 方向: PM递减是下游
                # 下游 = 索引更小 (PM更小)
                # 上游 = 索引更大 (PM更大)
                downstream_g_idx = ml_hv_group_indices[i - 1] if i > 0 else None
                upstream_g_idx = ml_hv_group_indices[i + 1] if i < len(ml_hv_group_indices) - 1 else None
            
            # 当前组中的 ML/HV 站点
            current_ml_hv = [idx for idx in g_stations 
                            if sorted_group.iloc[idx]['Type'] in ['ML', 'HV']]
            
            for station_idx in current_ml_hv:
                station = sorted_group.iloc[station_idx]
                
                # 连接下游（当前站点 → 下游站点）
                if downstream_g_idx is not None:
                    ds_pm, ds_stations = location_groups[downstream_g_idx]
                    # PM 差值初筛
                    if abs(ds_pm - station['Abs_PM']) <= max_pm_distance:
                        for target_idx in ds_stations:
                            target = sorted_group.iloc[target_idx]
                            if target['Type'] in ['ML', 'HV']:
                                edge_record, is_valid = create_edge_with_osrm(
                                    station, target, fwy, direction,
                                    'backbone', g_idx, downstream_g_idx
                                )
                                if is_valid:
                                    all_edges.append(edge_record)
                                else:
                                    rejected_edges.append(edge_record)
    
    edges_df = pd.DataFrame(all_edges)
    groups_df = pd.DataFrame(all_groups)
    rejected_df = pd.DataFrame(rejected_edges)
    
    # 去重
    if len(edges_df) > 0:
        edges_df = edges_df.drop_duplicates(subset=['Source_ID', 'Target_ID'])
    
    return edges_df, groups_df, rejected_df


# 构建第一层
print("\n开始构建第一层...")
layer1_edges, groups_df, layer1_rejected = build_layer1_backbone(
    meta_valid, MAX_PM_DISTANCE, check_osrm_colocation=False
)

print(f"\n第一层结果 (ML+HV 骨干网络):")
print(f"  有效边数: {len(layer1_edges)}")
print(f"  被拒绝边数 (OSRM距离过大): {len(layer1_rejected)}")
print(f"  位置组数: {len(groups_df)}")
print(f"  包含 ML 的位置组: {groups_df['Has_ML'].sum()}")
print(f"  包含 HV 的位置组: {groups_df['Has_HV'].sum()}")

if len(layer1_edges) > 0:
    print(f"\n边类型组合:")
    print(layer1_edges.groupby(['Source_Type', 'Target_Type']).size())
    
    print(f"\nOSRM 距离统计:")
    print(layer1_edges['OSRM_Distance'].describe())


开始构建第一层...


Layer1: ML+HV骨干:   0%|          | 0/1 [00:00<?, ?it/s]

Layer1: ML+HV骨干: 100%|██████████| 1/1 [00:01<00:00,  1.98s/it]


第一层结果 (ML+HV 骨干网络):
  有效边数: 131
  被拒绝边数 (OSRM距离过大): 3
  位置组数: 117
  包含 ML 的位置组: 72
  包含 HV 的位置组: 27

边类型组合:
Source_Type  Target_Type
HV           HV             19
             ML             25
ML           HV             24
             ML             63
dtype: int64

OSRM 距离统计:
count    131.000000
mean       0.668848
std        0.731731
min        0.002361
25%        0.208067
50%        0.475226
75%        0.706874
max        3.797271
Name: OSRM_Distance, dtype: float64


In [29]:
# 检查骨干网络连通性
def check_connectivity(edges_df, meta_df, station_types=['ML', 'HV']):
    """检查指定类型站点的连通性"""
    target_stations = set(meta_df[meta_df['Type'].isin(station_types)]['ID'])
    
    if len(edges_df) == 0:
        return {'total': len(target_stations), 'connected': 0, 'isolated': len(target_stations)}
    
    connected = set(edges_df['Source_ID']) | set(edges_df['Target_ID'])
    connected_target = connected & target_stations
    isolated = target_stations - connected_target
    
    return {
        'total': len(target_stations),
        'connected': len(connected_target),
        'isolated': len(isolated),
        'isolated_ids': isolated
    }


backbone_connectivity = check_connectivity(layer1_edges, meta_valid, ['ML', 'HV'])
print(f"\n骨干网络连通性:")
print(f"  ML+HV 总站点: {backbone_connectivity['total']}")
print(f"  有边的站点: {backbone_connectivity['connected']}")
print(f"  孤立站点: {backbone_connectivity['isolated']}")

if backbone_connectivity['isolated'] > 0:
    print(f"\n孤立站点 ID (前10):")
    for sid in list(backbone_connectivity['isolated_ids'])[:10]:
        s = meta_valid[meta_valid['ID'] == sid].iloc[0]
        print(f"  {sid} ({s['Type']}) Fwy={s['Fwy']}-{s['Dir']} PM={s['Abs_PM']:.3f}")


骨干网络连通性:
  ML+HV 总站点: 103
  有边的站点: 101
  孤立站点: 2

孤立站点 ID (前10):
  3415051 (ML) Fwy=99-N PM=323.372
  3415064 (ML) Fwy=99-N PM=334.100


## 6. 第二层：OR/FR 连接

In [30]:
def build_layer2_ramps(meta_df, check_osrm_colocation=False):
    """
    第二层：构建 OR/FR 连接
    
    OR (入口) 规则:
    1. 向下游搜索位置组
    2. 对于每个位置组:
       - 连接该组中的 ML, HV（如果有）
       - 连接该组中的 FR（如果有，且无 ML）
       - 跳过 OR（OR 不接受入边）
       - 如果该组有 ML，停止搜索
       - 否则继续搜索下一组
    
    FR (出口) 规则:
    1. 向上游搜索位置组
    2. 对于每个位置组:
       - 接受该组中 ML, HV 的入边（如果有）
       - 接受该组中 OR 的入边（如果有，且无 ML）
       - 跳过 FR（FR 不产生出边）
       - 如果该组有 ML，停止搜索
       - 否则继续搜索上一组
    
    方向处理:
    - N/E 方向: abs_pm 递增是下游
    - S/W 方向: abs_pm 递减是下游
    """
    all_edges = []
    rejected_edges = []
    
    for (fwy, direction), fwy_group in tqdm(meta_df.groupby(['Fwy', 'Dir']), desc="Layer2: OR/FR连接"):
        sorted_group = fwy_group.sort_values('Abs_PM').reset_index(drop=True)
        n = len(sorted_group)
        
        if n < 2:
            continue
        
        # 判断下游方向
        pm_increasing_is_downstream = is_downstream_direction(direction)
        
        # 划分同位置组（严格定义）
        location_groups = build_location_groups_strict(sorted_group, check_osrm=check_osrm_colocation)
        n_groups = len(location_groups)
        
        # 处理每个位置组
        for g_idx, (g_pm, g_stations) in enumerate(location_groups):
            
            # ========== 处理 OR (入口) ==========
            or_stations = [idx for idx in g_stations 
                          if sorted_group.iloc[idx]['Type'] == 'OR']
            
            for or_idx in or_stations:
                or_station = sorted_group.iloc[or_idx]
                
                # 根据方向确定下游搜索范围
                if pm_increasing_is_downstream:
                    # N/E: 下游是索引更大的组
                    downstream_range = range(g_idx + 1, n_groups)
                else:
                    # S/W: 下游是索引更小的组
                    downstream_range = range(g_idx - 1, -1, -1)
                
                # 向下游搜索
                for ds_g_idx in downstream_range:
                    ds_pm, ds_stations = location_groups[ds_g_idx]
                    
                    found_ml = False
                    
                    for target_idx in ds_stations:
                        target = sorted_group.iloc[target_idx]
                        target_type = target['Type']
                        
                        # OR 不接受入边
                        if target_type == 'OR':
                            continue
                        
                        # 可以连接: ML, HV, FR
                        if target_type in ['ML', 'HV', 'FR']:
                            edge_record, is_valid = create_edge_with_osrm(
                                or_station, target, fwy, direction,
                                'or_connect', g_idx, ds_g_idx
                            )
                            if is_valid:
                                all_edges.append(edge_record)
                            else:
                                rejected_edges.append(edge_record)
                        
                        if target_type == 'ML':
                            found_ml = True
                    
                    # 如果找到 ML，停止搜索
                    if found_ml:
                        break
            
            # ========== 处理 FR (出口) ==========
            fr_stations = [idx for idx in g_stations 
                          if sorted_group.iloc[idx]['Type'] == 'FR']
            
            for fr_idx in fr_stations:
                fr_station = sorted_group.iloc[fr_idx]
                
                # 根据方向确定上游搜索范围
                if pm_increasing_is_downstream:
                    # N/E: 上游是索引更小的组
                    upstream_range = range(g_idx - 1, -1, -1)
                else:
                    # S/W: 上游是索引更大的组
                    upstream_range = range(g_idx + 1, n_groups)
                
                # 向上游搜索
                for us_g_idx in upstream_range:
                    us_pm, us_stations = location_groups[us_g_idx]
                    
                    found_ml = False
                    
                    for source_idx in us_stations:
                        source = sorted_group.iloc[source_idx]
                        source_type = source['Type']
                        
                        # FR 不产生出边
                        if source_type == 'FR':
                            continue
                        
                        # 可以接受: ML, HV, OR
                        if source_type in ['ML', 'HV', 'OR']:
                            edge_record, is_valid = create_edge_with_osrm(
                                source, fr_station, fwy, direction,
                                'fr_connect', us_g_idx, g_idx
                            )
                            if is_valid:
                                all_edges.append(edge_record)
                            else:
                                rejected_edges.append(edge_record)
                        
                        if source_type == 'ML':
                            found_ml = True
                    
                    # 如果找到 ML，停止搜索
                    if found_ml:
                        break
    
    edges_df = pd.DataFrame(all_edges)
    rejected_df = pd.DataFrame(rejected_edges)
    
    # 去重
    if len(edges_df) > 0:
        edges_df = edges_df.drop_duplicates(subset=['Source_ID', 'Target_ID'])
    
    return edges_df, rejected_df


# 构建第二层
print("\n开始构建第二层...")
layer2_edges, layer2_rejected = build_layer2_ramps(meta_valid, check_osrm_colocation=False)

print(f"\n第二层结果 (OR/FR 连接):")
print(f"  有效边数: {len(layer2_edges)}")
print(f"  被拒绝边数 (OSRM距离过大): {len(layer2_rejected)}")

if len(layer2_edges) > 0:
    print(f"\n边类型组合:")
    print(layer2_edges.groupby(['Source_Type', 'Target_Type']).size())
    
    print(f"\n边类型分布:")
    print(layer2_edges['Edge_Type'].value_counts())
    
    print(f"\nOSRM 距离统计:")
    print(layer2_edges['OSRM_Distance'].describe())


开始构建第二层...


Layer2: OR/FR连接:   0%|          | 0/1 [00:00<?, ?it/s]

Layer2: OR/FR连接: 100%|██████████| 1/1 [00:01<00:00,  1.43s/it]


第二层结果 (OR/FR 连接):
  有效边数: 88
  被拒绝边数 (OSRM距离过大): 4

边类型组合:
Source_Type  Target_Type
HV           FR             11
ML           FR             17
OR           FR              5
             HV             19
             ML             36
dtype: int64

边类型分布:
Edge_Type
or_connect    60
fr_connect    28
Name: count, dtype: int64

OSRM 距离统计:
count    88.000000
mean      0.569578
std       0.723917
min       0.000932
25%       0.169806
50%       0.373259
75%       0.632480
max       3.835237
Name: OSRM_Distance, dtype: float64


In [31]:
# 合并第一层和第二层
layer12_edges = pd.concat([layer1_edges, layer2_edges], ignore_index=True)
layer12_edges = layer12_edges.drop_duplicates(subset=['Source_ID', 'Target_ID'])

print(f"\n第一层 + 第二层合并结果:")
print(f"  总边数: {len(layer12_edges)}")
print(f"\n边类型分布:")
print(layer12_edges['Edge_Type'].value_counts())

# 检查 OR/FR 连通性
or_connectivity = check_connectivity(layer12_edges, meta_valid, ['OR'])
fr_connectivity = check_connectivity(layer12_edges, meta_valid, ['FR'])

print(f"\nOR 连通性:")
print(f"  总站点: {or_connectivity['total']}")
print(f"  有边的站点: {or_connectivity['connected']}")
print(f"  孤立站点: {or_connectivity['isolated']}")

print(f"\nFR 连通性:")
print(f"  总站点: {fr_connectivity['total']}")
print(f"  有边的站点: {fr_connectivity['connected']}")
print(f"  孤立站点: {fr_connectivity['isolated']}")


第一层 + 第二层合并结果:
  总边数: 219

边类型分布:
Edge_Type
backbone      131
or_connect     60
fr_connect     28
Name: count, dtype: int64

OR 连通性:
  总站点: 36
  有边的站点: 34
  孤立站点: 2

FR 连通性:
  总站点: 18
  有边的站点: 16
  孤立站点: 2


In [32]:
# 验证 OR/FR 连边规则
def validate_or_fr_rules(edges_df):
    """验证 OR/FR 连边规则是否被遵守"""
    issues = []
    
    if len(edges_df) == 0:
        return issues
    
    # 规则1: OR 不应该有入边
    or_has_incoming = edges_df[edges_df['Target_Type'] == 'OR']
    if len(or_has_incoming) > 0:
        issues.append(f"OR 有 {len(or_has_incoming)} 条入边 (违规)")
    
    # 规则2: FR 不应该有出边
    fr_has_outgoing = edges_df[edges_df['Source_Type'] == 'FR']
    if len(fr_has_outgoing) > 0:
        issues.append(f"FR 有 {len(fr_has_outgoing)} 条出边 (违规)")
    
    return issues


issues = validate_or_fr_rules(layer12_edges)
if issues:
    print("\n⚠️ 发现规则违反:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("\n✓ OR/FR 连边规则验证通过")


✓ OR/FR 连边规则验证通过


## 7. 第三层：FF 连接（可选）

In [33]:
# FF 连接是可选的
ENABLE_FF_LAYER = False

if ENABLE_FF_LAYER:
    print("第三层 (FF 连接) 尚未实现")
    layer3_edges = pd.DataFrame()
else:
    print("第三层 (FF 连接) 已跳过")
    layer3_edges = pd.DataFrame()

第三层 (FF 连接) 已跳过


## 8. 合并所有层

In [34]:
# 合并所有层
all_layers = [layer1_edges, layer2_edges]
if len(layer3_edges) > 0:
    all_layers.append(layer3_edges)

final_edges = pd.concat(all_layers, ignore_index=True)
final_edges = final_edges.drop_duplicates(subset=['Source_ID', 'Target_ID'])

print(f"\n最终图结构:")
print(f"  总边数: {len(final_edges)}")
print(f"\n边类型分布:")
print(final_edges['Edge_Type'].value_counts())
print(f"\n站点类型组合:")
print(final_edges.groupby(['Source_Type', 'Target_Type']).size().sort_values(ascending=False))


最终图结构:
  总边数: 219

边类型分布:
Edge_Type
backbone      131
or_connect     60
fr_connect     28
Name: count, dtype: int64

站点类型组合:
Source_Type  Target_Type
ML           ML             63
OR           ML             36
HV           ML             25
ML           HV             24
OR           HV             19
HV           HV             19
ML           FR             17
HV           FR             11
OR           FR              5
dtype: int64


In [35]:
# 全局连通性检查
all_types = ['ML', 'HV', 'OR', 'FR', 'FF']
print("\n各类型站点连通性:")
print("="*50)

for stype in all_types:
    conn = check_connectivity(final_edges, meta_valid, [stype])
    if conn['total'] > 0:
        rate = conn['connected'] / conn['total'] * 100
        print(f"{stype}: {conn['connected']}/{conn['total']} ({rate:.1f}%)")


各类型站点连通性:
ML: 74/75 (98.7%)
HV: 28/28 (100.0%)
OR: 34/36 (94.4%)
FR: 16/18 (88.9%)
FF: 0/2 (0.0%)


## 9. 可视化（使用 OSRM 路径）

In [36]:
def visualize_layer_with_osrm_paths(edges_df, meta_df, output_path, title):
    """
    可视化图层，边使用 OSRM 实际路径
    
    颜色方案:
    - 站点: ML=蓝, HV=紫, OR=绿, FR=橙, FF=红
    - 边: backbone=蓝, or_connect=绿, fr_connect=橙, ff=红
    """
    # 边颜色
    edge_colors = {
        'backbone': '#2196F3',      # 蓝色
        'or_connect': '#4CAF50',    # 绿色
        'fr_connect': '#FF9800',    # 橙色
        'ff_same_fwy': '#9C27B0',   # 紫色
        'ff_cross_fwy': '#F44336',  # 红色
    }
    
    # 站点颜色
    station_colors = {
        'ML': '#2196F3',    # 蓝色
        'HV': '#9C27B0',    # 紫色
        'OR': '#4CAF50',    # 绿色
        'FR': '#FF9800',    # 橙色
        'FF': '#F44336',    # 红色
    }
    
    # 站点大小
    station_sizes = {
        'ML': 6,
        'HV': 5,
        'OR': 5,
        'FR': 5,
        'FF': 7,
    }
    
    # 计算地图中心
    center_lat = meta_df['Latitude'].mean()
    center_lon = meta_df['Longitude'].mean()
    
    # 创建地图
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=10,
        tiles='OpenStreetMap'
    )
    
    # 添加边（使用 OSRM 路径）
    if len(edges_df) > 0:
        for _, edge in edges_df.iterrows():
            color = edge_colors.get(edge['Edge_Type'], '#888888')
            
            # 尝试使用 OSRM 路径
            if pd.notna(edge.get('OSRM_Geometry')) and edge['OSRM_Geometry']:
                try:
                    geom = json.loads(edge['OSRM_Geometry'])
                    # GeoJSON 坐标是 [lon, lat]，需要转换为 [lat, lon]
                    coords = [[c[1], c[0]] for c in geom['coordinates']]
                    folium.PolyLine(
                        coords,
                        color=color,
                        weight=3,
                        opacity=0.8,
                        tooltip=f"{edge['Source_ID']} → {edge['Target_ID']} ({edge['Edge_Type']}, {edge['OSRM_Distance']:.2f}mi)"
                    ).add_to(m)
                except:
                    # 解析失败，使用直线
                    folium.PolyLine(
                        [[edge['Source_Lat'], edge['Source_Lon']],
                         [edge['Target_Lat'], edge['Target_Lon']]],
                        color=color,
                        weight=2,
                        opacity=0.6,
                        dash_array='5,5',  # 虚线表示无OSRM路径
                        tooltip=f"{edge['Source_ID']} → {edge['Target_ID']} ({edge['Edge_Type']}, no path)"
                    ).add_to(m)
            else:
                # 没有 OSRM 路径，使用直线（虚线）
                folium.PolyLine(
                    [[edge['Source_Lat'], edge['Source_Lon']],
                     [edge['Target_Lat'], edge['Target_Lon']]],
                    color=color,
                    weight=2,
                    opacity=0.6,
                    dash_array='5,5',
                    tooltip=f"{edge['Source_ID']} → {edge['Target_ID']} ({edge['Edge_Type']}, no path)"
                ).add_to(m)
    
    # 添加站点标记（在边之后，确保显示在上层）
    for _, station in meta_df.iterrows():
        color = station_colors.get(station['Type'], '#888888')
        size = station_sizes.get(station['Type'], 5)
        
        folium.CircleMarker(
            [station['Latitude'], station['Longitude']],
            radius=size,
            color='white',
            weight=1,
            fill=True,
            fillColor=color,
            fillOpacity=0.9,
            popup=folium.Popup(
                f"<b>{station['ID']}</b><br>"
                f"Type: {station['Type']}<br>"
                f"Fwy: {station['Fwy']}-{station['Dir']}<br>"
                f"PM: {station['Abs_PM']:.3f}<br>"
                f"Name: {station['Name']}",
                max_width=300
            ),
            tooltip=f"{station['ID']} ({station['Type']})"
        ).add_to(m)
    
    # 图例 HTML
    legend_html = f"""
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; 
                background-color: white; padding: 15px; border: 2px solid #333;
                border-radius: 8px; font-size: 12px; font-family: Arial;">
        <div style="font-weight: bold; margin-bottom: 10px; font-size: 14px;">{title}</div>
        
        <div style="font-weight: bold; margin-bottom: 5px;">站点类型:</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:{station_colors['ML']}; border-radius:50%; margin-right:5px;"></span>ML (主线)</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:{station_colors['HV']}; border-radius:50%; margin-right:5px;"></span>HV (HOV车道)</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:{station_colors['OR']}; border-radius:50%; margin-right:5px;"></span>OR (入口匝道)</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:{station_colors['FR']}; border-radius:50%; margin-right:5px;"></span>FR (出口匝道)</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:{station_colors['FF']}; border-radius:50%; margin-right:5px;"></span>FF (连接器)</div>
        
        <div style="font-weight: bold; margin-top: 10px; margin-bottom: 5px;">边类型:</div>
        <div><span style="display:inline-block; width:20px; height:3px; background:{edge_colors['backbone']}; margin-right:5px;"></span>backbone (骨干)</div>
        <div><span style="display:inline-block; width:20px; height:3px; background:{edge_colors['or_connect']}; margin-right:5px;"></span>or_connect (入口连接)</div>
        <div><span style="display:inline-block; width:20px; height:3px; background:{edge_colors['fr_connect']}; margin-right:5px;"></span>fr_connect (出口连接)</div>
        
        <div style="margin-top: 10px; font-size: 10px; color: #666;">
            实线: OSRM路径<br>
            虚线: 直线连接
        </div>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    
    m.save(output_path)
    print(f"地图已保存: {output_path}")
    return m


fwy_suffix = f"_{TARGET_FWY}" if TARGET_FWY else ""
print("可视化函数定义完成")

可视化函数定义完成


In [37]:
# 可视化第一层
print("生成第一层可视化...")
visualize_layer_with_osrm_paths(
    layer1_edges, 
    meta_valid[meta_valid['Type'].isin(['ML', 'HV'])],
    os.path.join(OUTPUT_DIR, f'layer1_backbone{fwy_suffix}.html'),
    '第一层: ML+HV 骨干网络'
)

生成第一层可视化...
地图已保存: ./output/layered_graph_v2/layer1_backbone_99.html


In [38]:
# 可视化第一层+第二层
print("生成第一层+第二层可视化...")
visualize_layer_with_osrm_paths(
    layer12_edges,
    meta_valid[meta_valid['Type'].isin(['ML', 'HV', 'OR', 'FR'])],
    os.path.join(OUTPUT_DIR, f'layer12_with_ramps{fwy_suffix}.html'),
    '第一层+第二层: 骨干网络 + OR/FR'
)

生成第一层+第二层可视化...
地图已保存: ./output/layered_graph_v2/layer12_with_ramps_99.html


In [39]:
# 可视化完整图
print("生成完整图可视化...")
visualize_layer_with_osrm_paths(
    final_edges,
    meta_valid,
    os.path.join(OUTPUT_DIR, f'final_graph{fwy_suffix}.html'),
    '完整交通流图'
)

生成完整图可视化...
地图已保存: ./output/layered_graph_v2/final_graph_99.html


## 10. 保存结果

In [40]:
# 保存边（不含 geometry，太大）
save_cols = [
    'Source_ID', 'Target_ID', 'Fwy', 'Dir',
    'Source_Type', 'Target_Type',
    'Source_PM', 'Target_PM', 'PM_Diff',
    'Source_Lat', 'Source_Lon', 'Target_Lat', 'Target_Lon',
    'Source_Name', 'Target_Name',
    'Edge_Type', 'Geo_Distance', 'OSRM_Distance'
]

# 保存各层边
if len(layer1_edges) > 0:
    layer1_edges[save_cols].to_csv(
        os.path.join(OUTPUT_DIR, f'layer1_backbone{fwy_suffix}.csv'), index=False
    )
    print(f"第一层边已保存")

if len(layer2_edges) > 0:
    layer2_edges[save_cols].to_csv(
        os.path.join(OUTPUT_DIR, f'layer2_ramps{fwy_suffix}.csv'), index=False
    )
    print(f"第二层边已保存")

# 保存最终图
if len(final_edges) > 0:
    final_edges[save_cols].to_csv(
        os.path.join(OUTPUT_DIR, f'final_edges{fwy_suffix}.csv'), index=False
    )
    print(f"最终边已保存")

# 保存被拒绝的边（用于分析）
all_rejected = pd.concat([layer1_rejected, layer2_rejected], ignore_index=True)
if len(all_rejected) > 0:
    all_rejected[save_cols].to_csv(
        os.path.join(OUTPUT_DIR, f'rejected_edges{fwy_suffix}.csv'), index=False
    )
    print(f"被拒绝边已保存")

# 保存位置组信息
groups_df.to_csv(
    os.path.join(OUTPUT_DIR, f'location_groups{fwy_suffix}.csv'), index=False
)
print(f"位置组信息已保存")

第一层边已保存
第二层边已保存
最终边已保存
被拒绝边已保存
位置组信息已保存


In [41]:
# 生成汇总报告
report = f"""
分层交通流图构建报告 V2
{'='*60}

配置:
  目标高速: {TARGET_FWY if TARGET_FWY else '全部'}
  最大 PM 距离 (初筛): {MAX_PM_DISTANCE} mi
  最大 OSRM 距离 (验证): {MAX_OSRM_DISTANCE} mi
  OSRM 服务器: {OSRM_SERVER}
  OSRM 可用: {OSRM_AVAILABLE}

同位置组定义:
  - Abs_PM 完全一致
  - 经纬度距离 < 0.01 mi
  - OSRM 距离 < 0.01 mi (可选)

数据概况:
  站点总数: {len(meta_valid)}
  ML 站点: {len(meta_valid[meta_valid['Type']=='ML'])}
  HV 站点: {len(meta_valid[meta_valid['Type']=='HV'])}
  OR 站点: {len(meta_valid[meta_valid['Type']=='OR'])}
  FR 站点: {len(meta_valid[meta_valid['Type']=='FR'])}
  FF 站点: {len(meta_valid[meta_valid['Type']=='FF'])}
  位置组数: {len(groups_df)}

第一层 (ML+HV 骨干网络):
  有效边数: {len(layer1_edges)}
  被拒绝边数: {len(layer1_rejected)}

第二层 (OR/FR 连接):
  有效边数: {len(layer2_edges)}
  被拒绝边数: {len(layer2_rejected)}

最终图:
  总边数: {len(final_edges)}

连通性:
  ML: {check_connectivity(final_edges, meta_valid, ['ML'])['connected']}/{check_connectivity(final_edges, meta_valid, ['ML'])['total']}
  HV: {check_connectivity(final_edges, meta_valid, ['HV'])['connected']}/{check_connectivity(final_edges, meta_valid, ['HV'])['total']}
  OR: {check_connectivity(final_edges, meta_valid, ['OR'])['connected']}/{check_connectivity(final_edges, meta_valid, ['OR'])['total']}
  FR: {check_connectivity(final_edges, meta_valid, ['FR'])['connected']}/{check_connectivity(final_edges, meta_valid, ['FR'])['total']}

输出文件:
  - layer1_backbone{fwy_suffix}.csv: 第一层边
  - layer2_ramps{fwy_suffix}.csv: 第二层边
  - final_edges{fwy_suffix}.csv: 最终边
  - rejected_edges{fwy_suffix}.csv: 被拒绝的边
  - layer1_backbone{fwy_suffix}.html: 第一层地图
  - layer12_with_ramps{fwy_suffix}.html: 第一+二层地图
  - final_graph{fwy_suffix}.html: 完整图地图
"""

report_file = os.path.join(OUTPUT_DIR, f'graph_report{fwy_suffix}.txt')
with open(report_file, 'w') as f:
    f.write(report)
print(f"报告已保存: {report_file}")
print(report)

报告已保存: ./output/layered_graph_v2/graph_report_99.txt

分层交通流图构建报告 V2

配置:
  目标高速: 99
  最大 PM 距离 (初筛): 4.0 mi
  最大 OSRM 距离 (验证): 4.0 mi
  OSRM 服务器: http://localhost:5000
  OSRM 可用: True

同位置组定义:
  - Abs_PM 完全一致
  - 经纬度距离 < 0.01 mi
  - OSRM 距离 < 0.01 mi (可选)

数据概况:
  站点总数: 159
  ML 站点: 75
  HV 站点: 28
  OR 站点: 36
  FR 站点: 18
  FF 站点: 2
  位置组数: 117

第一层 (ML+HV 骨干网络):
  有效边数: 131
  被拒绝边数: 3

第二层 (OR/FR 连接):
  有效边数: 88
  被拒绝边数: 4

最终图:
  总边数: 219

连通性:
  ML: 74/75
  HV: 28/28
  OR: 34/36
  FR: 16/18

输出文件:
  - layer1_backbone_99.csv: 第一层边
  - layer2_ramps_99.csv: 第二层边
  - final_edges_99.csv: 最终边
  - rejected_edges_99.csv: 被拒绝的边
  - layer1_backbone_99.html: 第一层地图
  - layer12_with_ramps_99.html: 第一+二层地图
  - final_graph_99.html: 完整图地图



## 总结

### V2 版本主要改进

| 项目 | V1 | V2 |
|------|----|----|  
| 同位置组 | PM差 < 0.1 mi | PM完全相同 + 坐标距离≈0 + OSRM距离≈0 |
| 连边验证 | 仅 PM 差值 | PM 差值 + OSRM 距离 |
| 可视化边 | 直线 | OSRM 实际路径 |
| 颜色 | 简单区分 | 精细区分站点和边类型 |

### 连边规则

| Source | Target | 允许 | 说明 |
|--------|--------|------|------|
| ML/HV | ML/HV | ✓ | 骨干网络 |
| ML/HV | FR | ✓ | 流向出口 |
| OR | ML/HV | ✓ | 从入口汇入 |
| OR | FR | ✓ | 编织区 |
| FR | * | ✗ | FR 不产生出边 |
| * | OR | ✗ | OR 不接受入边 |